# WeLoveReviews: exploración inicial de reseñas

## Objetivo de esta etapa

Antes de aplicar cualquier modelo, vamos a conocer el dataset de reseñas de Harbor House Café. En esta primera etapa comprobaremos su estructura, la calidad de los datos, la distribución de las puntuaciones y la longitud de los textos.

La pregunta de negocio es si el sentimiento de las reseñas escritas parece compatible con una puntuación media cercana a 4.5 estrellas. Todavía no ejecutaremos ningún modelo de sentimiento: primero entenderemos los datos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/raw/reviews.csv")
reviews = pd.read_csv(DATA_PATH)

reviews.head()

## 1. Estructura y calidad del dataset

Primero confirmamos el tamaño del dataset, los tipos de datos y la existencia de valores ausentes o duplicados.

In [ ]:
print(f"Filas: {reviews.shape[0]}")
print(f"Columnas: {reviews.shape[1]}")
print("\nTipos de datos:")
print(reviews.dtypes)
print("\nValores ausentes por columna:")
print(reviews.isna().sum())
print(f"\nFilas duplicadas: {reviews.duplicated().sum()}")
print(f"IDs duplicados: {reviews['review_id'].duplicated().sum()}")

## 2. Distribución de las puntuaciones

La puntuación media de referencia es 4.5 estrellas. Ahora observamos cómo se reparte esa puntuación entre las 500 reseñas.

In [ ]:
rating_counts = reviews["rating"].value_counts().sort_index()
rating_summary = pd.DataFrame({
    "reviews": rating_counts,
    "percentage": (rating_counts / len(reviews) * 100).round(1),
})

print(f"Puntuación media: {reviews['rating'].mean():.2f} / 5")
print(rating_summary)

plt.figure(figsize=(7, 4))
sns.countplot(data=reviews, x="rating", color="#4C78A8")
plt.title("Distribución de las puntuaciones")
plt.xlabel("Rating humano")
plt.ylabel("Número de reseñas")
plt.show()

## 3. Longitud de las reseñas

Medimos el número de palabras para comprobar si los textos tienen suficiente contexto y si existen valores extremos.

In [ ]:
reviews["word_count"] = reviews["review_text"].str.split().str.len()

print(reviews["word_count"].describe().round(2))

plt.figure(figsize=(8, 4))
sns.histplot(data=reviews, x="word_count", bins=15, color="#F58518")
plt.title("Longitud de las reseñas")
plt.xlabel("Número de palabras")
plt.ylabel("Número de reseñas")
plt.show()

## 4. Muestra de textos

Finalmente, revisamos ejemplos reales para entender el tono y el tipo de lenguaje utilizado por los clientes.

In [ ]:
print(reviews[["review_id", "rating", "review_text"]].sample(5, random_state=42).to_string(index=False))

## 5. Insights accionables

Los principales hallazgos de esta exploración son:

- El dataset contiene 500 reseñas y tres columnas esperadas: identificador, puntuación y texto.
- La puntuación media es exactamente 4.5 estrellas. Hay un claro predominio de puntuaciones de 5 estrellas: 360 de 500 reseñas (72%).
- También existe una minoría relevante de reseñas de 1 a 3 estrellas (68 reseñas, 13.6%), que conviene comparar con el sentimiento escrito.
- Los textos tienen una longitud suficiente para aportar contexto: una mediana de 29 palabras, con valores entre 14 y 42 palabras.
- Las reseñas mezclan valoraciones generales con detalles sobre café, comida, servicio y ambiente. Esto hace especialmente importante comprobar si una queja puntual cambia la predicción del modelo.

## 6. Propuesta de limpieza

No se necesita una limpieza agresiva. No hay valores ausentes, filas duplicadas ni identificadores repetidos, y las columnas tienen los tipos esperados.

Conservaremos el texto completo porque palabras de contexto como `pero`, `aunque` o `no` pueden cambiar el significado de una reseña. No modificaremos el texto antes de aplicar el modelo.

Esta etapa termina con la exploración y la propuesta de limpieza. El modelado y la comparación entre estrellas y sentimiento se realizarán después, en las siguientes secciones del proyecto.

## 7. Plan de acción y justificación del modelo

La exploración confirma que tenemos 500 textos completos, sin valores ausentes ni duplicados, y con una distribución de ratings muy concentrada en 4 y 5 estrellas. Por ello, el siguiente paso será comparar la puntuación humana con el sentimiento que se desprende del texto.

### Plan

1. Cargar una única vez el modelo preentrenado de Hugging Face.
2. Procesar las 500 reseñas y guardar la predicción de estrellas para cada una.
3. Convertir la predicción a una banda de sentimiento:
   - 1–2 estrellas: negativo.
   - 3 estrellas: neutral.
   - 4–5 estrellas: positivo.
4. Comparar las bandas predichas con el rating humano y con la media de 4.5 estrellas.
5. Revisar manualmente una muestra de 15–20 reseñas, prestando especial atención a los desacuerdos.

### Modelo elegido

Utilizaremos `nlptown/bert-base-multilingual-uncased-sentiment`. Es adecuado como primera aproximación porque ya está ajustado para interpretar reseñas en varios idiomas y devuelve una valoración de 1 a 5 estrellas.

No entrenaremos un modelo desde cero. La integración se hará mediante `pipeline()` y el modelo se cargará una sola vez, antes de la inferencia. En la fase de implementación fijaremos explícitamente el identificador y la revisión del modelo, además de las versiones de las dependencias, para que otra ejecución no dependa de cambios posteriores en Hugging Face.

### Limitación que debemos comprobar

El modelo fue entrenado principalmente con reseñas de productos, mientras que nuestro dataset contiene reseñas de servicios. Una queja sobre la espera, el personal o el ambiente puede aparecer junto a una valoración general positiva. Por ese desajuste esperamos posibles falsos negativos: reseñas con 4 o 5 estrellas que el modelo clasifique como negativas.

Por ese motivo no interpretaremos la predicción como una verdad absoluta. La utilizaremos como una segunda señal, la compararemos con las estrellas y revisaremos ejemplos concretos antes de extraer conclusiones para la account manager.

Esta sección define el plan. La inferencia y sus resultados se añadirán en el siguiente paso.

## 8. Inferencia de sentimiento

Ahora aplicamos el modelo fijado a las 500 reseñas. El pipeline se crea una sola vez y después se reutiliza para todas las reseñas.

In [ ]:
from transformers import pipeline

MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
MODEL_REVISION = "8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d"

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    revision=MODEL_REVISION,
)

predictions = sentiment_pipeline(
    reviews["review_text"].tolist(),
    truncation=True,
    batch_size=16,
)

reviews["predicted_rating"] = [int(prediction["label"].split()[0]) for prediction in predictions]

sentiment_map = {
    1: "negative",
    2: "negative",
    3: "neutral",
    4: "positive",
    5: "positive",
}
reviews["sentiment"] = reviews["predicted_rating"].map(sentiment_map)

reviews[["review_id", "rating", "predicted_rating", "sentiment", "review_text"]].head()

## 9. Distribución del sentimiento predicho

Convertimos las predicciones de estrellas en porcentajes sencillos para la account manager y las comparamos con la distribución humana.

In [ ]:
sentiment_summary = (
    reviews["sentiment"]
    .value_counts()
    .reindex(["positive", "neutral", "negative"], fill_value=0)
    .rename_axis("sentiment")
    .to_frame("reviews")
)
sentiment_summary["percentage"] = (
    sentiment_summary["reviews"] / len(reviews) * 100
).round(1)

print(sentiment_summary)
print("\nComparación con los ratings humanos:")
print(pd.crosstab(reviews["rating"], reviews["sentiment"], normalize="index").round(3))

plt.figure(figsize=(7, 4))
sns.countplot(
    data=reviews,
    x="sentiment",
    order=["positive", "neutral", "negative"],
    palette=["#54A24B", "#B279A2", "#E45756"],
    hue="sentiment",
    legend=False,
)
plt.title("Distribución del sentimiento predicho")
plt.xlabel("Banda de sentimiento")
plt.ylabel("Número de reseñas")
plt.show()